In [46]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing import image
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization

In [30]:
data_df = pd.read_csv("HAM10000_metadata.csv")  # ضع هنا مسار CSV الخاص بك
print(data_df.head())
print(data_df['dx'].value_counts()) 

     lesion_id      image_id   dx dx_type   age   sex localization
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


In [31]:
def get_all_image_paths(folders):
    image_paths = {}
    for folder in folders:
        for subdir, _, files in os.walk(folder):
            for file in files:
                if file.endswith(".jpg"):
                    img_id = os.path.splitext(file)[0]
                    image_paths[img_id] = os.path.join(subdir, file)
    return image_paths

folders = ["E:\project\project CNN,RNN,Transformer\HAM10000_images_part_1", "E:\project\project CNN,RNN,Transformer\HAM10000_images_part_2"]  # ضع هنا مسارات الفولدرين
image_dict = get_all_image_paths(folders)


<>:11: SyntaxWarning: invalid escape sequence '\p'
<>:11: SyntaxWarning: invalid escape sequence '\p'
<>:11: SyntaxWarning: invalid escape sequence '\p'
<>:11: SyntaxWarning: invalid escape sequence '\p'
C:\Users\saram\AppData\Local\Temp\ipykernel_20764\3188868253.py:11: SyntaxWarning: invalid escape sequence '\p'
  folders = ["E:\project\project CNN,RNN,Transformer\HAM10000_images_part_1", "E:\project\project CNN,RNN,Transformer\HAM10000_images_part_2"]  # ضع هنا مسارات الفولدرين
C:\Users\saram\AppData\Local\Temp\ipykernel_20764\3188868253.py:11: SyntaxWarning: invalid escape sequence '\p'
  folders = ["E:\project\project CNN,RNN,Transformer\HAM10000_images_part_1", "E:\project\project CNN,RNN,Transformer\HAM10000_images_part_2"]  # ضع هنا مسارات الفولدرين


In [32]:
IMG_SIZE = 128  # حجم الصور للتدريب
def load_images_from_dict(df, image_dict, img_size=IMG_SIZE):
    images = []
    valid_indices = []
    for idx, img_id in enumerate(df['image_id']):
        if img_id in image_dict:
            img_path = image_dict[img_id]
            img = image.load_img(img_path, target_size=(img_size, img_size))
            img = image.img_to_array(img) / 255.0  # normalize
            images.append(img)
            valid_indices.append(idx)
        else:
            print(f"Image not found: {img_id}")
    return np.array(images), df.iloc[valid_indices]

X, data_df = load_images_from_dict(data_df, image_dict)
y = pd.get_dummies(data_df['dx']).values

print(f"Total images loaded: {X.shape[0]}")

Total images loaded: 10015


In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [50]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True
)
datagen.fit(X_train)


In [44]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(2,2),
    
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(y.shape[1], activation='softmax')  # عدد العقد يساوي عدد الأصناف
])




c:\Users\saram\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [47]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_13 (Conv2D)              │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,517,831 (24.86 MB)

 Trainable params: 6,517,831 (24.86 MB)

 Non-trainable params: 0 (0.00 B)

In [51]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    validation_data=(X_test, y_test),
    epochs=30,
    callbacks=[early_stop, reduce_lr]
)


Epoch 1/30
251/251 ━━━━━━━━━━━━━━━━━━━━ 62s 247ms/step - accuracy: 0.6695 - loss: 1.0209 - val_accuracy: 0.6695 - val_loss: 0.9307 - learning_rate: 2.0000e-04
Epoch 2/30
251/251 ━━━━━━━━━━━━━━━━━━━━ 68s 272ms/step - accuracy: 0.6695 - loss: 0.9372 - val_accuracy: 0.6695 - val_loss: 0.8686 - learning_rate: 2.0000e-04
Epoch 3/30
251/251 ━━━━━━━━━━━━━━━━━━━━ 70s 279ms/step - accuracy: 0.6695 - loss: 0.8905 - val_accuracy: 0.6705 - val_loss: 0.8397 - learning_rate: 2.0000e-04
Epoch 4/30
251/251 ━━━━━━━━━━━━━━━━━━━━ 72s 288ms/step - accuracy: 0.6832 - loss: 0.8489 - val_accuracy: 0.6805 - val_loss: 0.8095 - learning_rate: 2.0000e-04
Epoch 5/30
251/251 ━━━━━━━━━━━━━━━━━━━━ 73s 290ms/step - accuracy: 0.6918 - loss: 0.8134 - val_accuracy: 0.6900 - val_loss: 0.7843 - learning_rate: 2.0000e-04
Epoch 6/30
251/251 ━━━━━━━━━━━━━━━━━━━━ 75s 299ms/step - accuracy: 0.7034 - loss: 0.7930 - val_accuracy: 0.6985 - val_loss: 0.7726 - learning_rate: 2.0000e-04
Epoch 7/30
251/251 ━━━━━━━━━━━━━━━━━━━━ 75s 29

In [52]:
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {acc*100:.2f}%")


63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 57ms/step - accuracy: 0.7434 - loss: 0.6636
Test Accuracy: 74.34%
